# Warm-User Long-Tail Evaluation
**Re-runs the warm evaluation from `final_evaluation.ipynb` with the top-100
globally-popular songs excluded from each user's held-out ground truth.**

Addresses the Steck (2011) popularity bias: mega-hits inflate hit-rate metrics
for non-personalized models. Exclusion isolates each model's ability to recommend
beyond the obvious.

All 1000 test users, split, and model objects loaded from existing caches.

---
## Section 1 — Setup

In [ ]:
import os, sys
os.environ['OPENBLAS_NUM_THREADS'] = '1'

from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import joblib
from scipy.sparse import csr_matrix

from src.data import (
    load_msd_metadata, load_spotify_tracks, match_datasets, build_metadata_catalog,
)
from src.models import CFModel, ContentModel, PopularityBaseline
from src.hybrid import HybridRecommender
from src.evaluation import evaluate_model, bootstrap_ci, paired_bootstrap_test

MODELS_DIR  = Path('../models')
RESULTS_DIR = Path('../results')
K_VALUES    = [5, 10, 20]


In [ ]:
# Load all models from existing pkl artifacts
cf      = CFModel.load(MODELS_DIR)
cm      = ContentModel.load(MODELS_DIR)
triplets = joblib.load(RESULTS_DIR / 'triplets_cache.pkl')
pop     = PopularityBaseline().fit(triplets)

msd_meta = load_msd_metadata()
spotify  = load_spotify_tracks()
matched  = match_datasets(spotify, msd_meta)
meta_cat = build_metadata_catalog(matched)

hybrid = HybridRecommender(cf, cm, meta_cat, alpha_strategy='adaptive')
cf_song_to_idx = {sid: idx for idx, sid in cf._idx_to_song.items()}
print('Models loaded')


In [ ]:
# Load warm-eval split from final_evaluation.ipynb cache
split                  = joblib.load(RESULTS_DIR / 'eval_split.pkl')
sampled_users          = split['sampled_users']
test_data              = split['test_data']
user_train_songs       = split['user_train_songs']
user_most_played_in_ct = split['user_most_played_in_ct']
print(f'Loaded split: {len(sampled_users)} users')
print(f'Users with content seed: {len(user_most_played_in_ct)}')


---
## Section 2 — Long-Tail Ground Truth

In [ ]:
# Top-100 globally most-played songs by total play count
top100_global = set(
    triplets.groupby('sid')['count'].sum()
    .sort_values(ascending=False)
    .head(100)
    .index.astype(str)
)
print(f'Top-100 computed. Examples: {list(top100_global)[:3]}')

# Filter each user's held-out set; drop users left with empty sets
test_data_lt = {}
for uid, held_out in test_data.items():
    filtered = held_out - top100_global
    if filtered:
        test_data_lt[uid] = filtered

dropped = len(test_data) - len(test_data_lt)
print(f'Long-tail users: {len(test_data_lt)} retained, {dropped} dropped')
print(f'Median held-out (LT): {int(np.median([len(v) for v in test_data_lt.values()]))}')


---
## Section 3 — Evaluation

In [ ]:
# Same recommend_fns as final_evaluation.ipynb
def cf_fn(user_id, k):
    uidx      = cf._user_to_idx[user_id]
    full      = cf._user_item[uidx].toarray().flatten()
    mask      = np.zeros(full.shape, dtype=bool)
    for sid in user_train_songs[user_id]:
        if sid in cf_song_to_idx:
            mask[cf_song_to_idx[sid]] = True
    train_row = csr_matrix(full * mask)
    idxs, _   = cf._model.recommend(uidx, train_row, N=k,
                                     filter_already_liked_items=True)
    return [cf._idx_to_song[int(i)] for i in idxs]

def content_fn(user_id, k):
    seed = user_most_played_in_ct.get(user_id)
    if seed is None:
        return []
    return cm.recommend(seed, k=k)['song_id'].tolist()

def pop_fn(user_id, k):
    return pop.recommend(k=k)['song_id'].tolist()

def hybrid_fn(user_id, k):
    recs = hybrid.recommend(
        user_id=user_id, k=k,
        known_song_ids=set(user_train_songs[user_id]),
    )
    return recs['song_id'].tolist()


In [ ]:
_WLT_PER_USER = RESULTS_DIR / 'warm_longtail_per_user.csv'

if _WLT_PER_USER.exists():
    per_user_wlt = pd.read_csv(_WLT_PER_USER)
    print(f'Loaded from cache ({len(per_user_wlt):,} rows)')
else:
    model_fns = {
        'popularity': pop_fn,
        'cf':         cf_fn,
        'content':    content_fn,
        'hybrid':     hybrid_fn,
    }
    frames = []
    for name, fn in model_fns.items():
        print(f'Evaluating {name}...', end=' ', flush=True)
        df = evaluate_model(fn, test_data_lt, k_values=K_VALUES)
        df['model'] = name
        frames.append(df)
        print('done')
    per_user_wlt = pd.concat(frames, ignore_index=True)
    per_user_wlt.to_csv(_WLT_PER_USER, index=False)
    print(f'Saved ({len(per_user_wlt):,} rows)')


---
## Section 4 — Metrics + Hypothesis Tests

In [ ]:
MODEL_ORDER  = ['popularity', 'cf', 'content', 'hybrid']
METRIC_ORDER = ['ndcg', 'hit_rate', 'recall', 'precision']

rows = []
for model in MODEL_ORDER:
    sub = per_user_wlt[per_user_wlt['model'] == model]
    for k in K_VALUES:
        for metric in METRIC_ORDER:
            scores = sub[(sub['k'] == k) & (sub['metric'] == metric)]['value'].values
            point, lo, hi = bootstrap_ci(scores)
            rows.append({'model': model, 'k': k, 'metric': metric,
                         'mean': point, 'ci_low': lo, 'ci_high': hi})

wlt_metrics = pd.DataFrame(rows)
wlt_metrics.to_csv(RESULTS_DIR / 'warm_longtail_metrics.csv', index=False)

n_lt = len(test_data_lt)
print(f'### Warm Long-Tail Metrics @ k=10 (top-100 excluded, n={n_lt} users)\n')
print(f'{"Model":<12} {"NDCG@10_LT":<24} {"HitRate@10_LT":<24} {"Recall@10_LT":<24}')
print('-' * 85)
for model in MODEL_ORDER:
    parts = [f'{model:<12}']
    for metric in ['ndcg', 'hit_rate', 'recall']:
        r = wlt_metrics[(wlt_metrics['model']==model) &
                        (wlt_metrics['k']==10) &
                        (wlt_metrics['metric']==metric)].iloc[0]
        parts.append(f'{r["mean"]:.4f} [{r["ci_low"]:.4f},{r["ci_high"]:.4f}]')
    print('  '.join(f'{p:<24}' for p in parts))


In [ ]:
def get_ndcg10(df, model):
    return (df[(df['model']==model) & (df['k']==10) & (df['metric']=='ndcg')]
            .sort_values('user_id')['value'].values)

# Align on users present in LT test data
lt_users = set(test_data_lt.keys())
per_lt   = per_user_wlt[per_user_wlt['user_id'].isin(lt_users)]

hy  = get_ndcg10(per_lt, 'hybrid')
cf_ = get_ndcg10(per_lt, 'cf')
pp  = get_ndcg10(per_lt, 'popularity')
ct  = get_ndcg10(per_lt, 'content')

comparisons = [
    ('Hybrid vs CF',         hy, cf_),
    ('Hybrid vs Popularity', hy, pp),
    ('Hybrid vs Content',    hy, ct),
]

ht_rows = []
print(f'{"Comparison":<25} {"delta_mean":>12} {"p_value":>10} {"sig@0.05":>10}')
print('-' * 62)
for label, a, b in comparisons:
    delta = float((a - b).mean())
    pval  = paired_bootstrap_test(a, b, n_resamples=10_000, seed=42)
    sig   = pval < 0.05
    ht_rows.append({'comparison': label, 'delta_mean': delta,
                    'p_value': pval, 'significant_at_0.05': sig})
    print(f'{label:<25} {delta:>12.5f} {pval:>10.4f} {str(sig):>10}')

pd.DataFrame(ht_rows).to_csv(RESULTS_DIR / 'warm_longtail_hypothesis.csv', index=False)
print('\nSaved warm_longtail_metrics.csv + warm_longtail_hypothesis.csv')

# Compare to standard warm ranking
warm_std = pd.read_csv(RESULTS_DIR / 'final_metrics.csv')
print('\n--- NDCG@10: standard warm vs long-tail warm ---')
for model in MODEL_ORDER:
    std_r = warm_std[(warm_std['model']==model)&(warm_std['k']==10)&(warm_std['metric']=='ndcg')].iloc[0]
    lt_r  = wlt_metrics[(wlt_metrics['model']==model)&(wlt_metrics['k']==10)&(wlt_metrics['metric']=='ndcg')].iloc[0]
    print(f'  {model:<12} std={std_r["mean"]:.4f}  lt={lt_r["mean"]:.4f}')
